# Lab 3: SQL: Asking the Company's Data
**BUS-G 492 · Coding with AI for Business Analysts**

> **INSTRUCTOR VERSION — contains solutions. Do not distribute.**

In [1]:
# --- Setup: run this cell first -------------------------------------------
# Finds the course data whether you are in Colab or on your own laptop.
# INSTRUCTOR: set DATA_URL once to the raw GitHub folder that holds /data (see instructor README).
import os
DATA_URL = "https://raw.githubusercontent.com/ilovemypinktutu/g492-coding/main/data/"
for candidate in ["data/", "../data/", "../../data/"]:
    if os.path.exists(candidate + "orders.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: ../../data/


**Goal today:** write, read and *check* SQL. In most companies the data lives in a database or a warehouse, and SQL is how you ask it questions.

We use **DuckDB**, a real SQL engine that runs inside this notebook. The SQL you write here works almost unchanged in Snowflake, BigQuery, Postgres or SQL Server.

In [2]:
# %pip install -q duckdb     # uncomment if duckdb is not installed
import duckdb, pandas as pd
con = duckdb.connect()
for t in ["orders", "order_items", "products", "customers", "returns", "fx_rates_monthly"]:
    con.register(t + "_df", pd.read_csv(DATA + t + ".csv"))
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM {t}_df")
    con.unregister(t + "_df")
def q(sql):
    """Run a SQL query and show the result as a table."""
    return con.sql(sql).df()
q("SHOW TABLES")

,name
0,customers
1,fx_rates_monthly
2,order_items
3,orders
4,products
5,returns


## Part 1: The six clauses (10 min)
```sql
SELECT   columns / calculations      -- what to show
FROM     table                       -- where from
JOIN     other ON key                -- look up another table
WHERE    condition                   -- keep some rows (before grouping)
GROUP BY columns                     -- PivotTable rows
HAVING   condition                   -- keep some groups (after grouping)
ORDER BY column DESC LIMIT n         -- sort, top n
```
**1a.** Show the 10 most recent orders from the United Kingdom.

In [3]:
q("""
SELECT order_id, order_date, customer_id, channel, status
FROM orders
WHERE country = 'United Kingdom'
ORDER BY order_date DESC
LIMIT 10
""")

,order_id,order_date,customer_id,channel,status
0,520673,2026-06-30,10581,Web,Completed
1,520678,2026-06-30,12226,Mobile app,Completed
2,520685,2026-06-30,10461,Mobile app,Completed
3,520695,2026-06-30,10428,Mobile app,Completed
4,520701,2026-06-30,11711,Web,Completed
5,520702,2026-06-30,12657,Web,Completed
6,520704,2026-06-30,12945,Web,Completed
7,520718,2026-06-30,12226,Web,Completed
8,520720,2026-06-30,11592,Web,Completed
9,520619,2026-06-29,12226,Web,Completed


**1b.** For each country: number of orders, number of *completed* orders, and cancellation rate (%). Sort by orders, descending.

In [4]:
q("""
SELECT country,
       COUNT(*) AS orders,
       SUM(CASE WHEN status = 'Completed' THEN 1 ELSE 0 END) AS completed,
       ROUND(100.0 * AVG(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END), 1) AS cancel_rate_pct
FROM orders
GROUP BY country
ORDER BY orders DESC
""")

,country,orders,completed,cancel_rate_pct
0,United States,11317,10792.0,4.6
1,United Kingdom,3403,3258.0,4.3
2,Germany,2899,2767.0,4.6
3,France,1381,1325.0,4.1
4,Netherlands,874,840.0,3.9
5,Ireland,850,816.0,4.0


## Part 2: Text-to-SQL with AI, then verify (20 min)
Copy the schema printed below into your AI assistant, followed by the question. **Always give the AI the schema**, or it will make up column names.

In [5]:
schema = "\n".join(
    f"{t}({', '.join(c + ' ' + str(ty) for c, ty in con.sql(f'SELECT * FROM {t} LIMIT 0').df().dtypes.items())})"
    for t in ["orders", "order_items", "products", "customers", "returns", "fx_rates_monthly"])
print(schema)

orders(order_id int64, order_date object, customer_id int64, country object, currency object, channel object, status object)
order_items(order_id int64, line_no int64, product_id int64, quantity int64, unit_price float64, discount_pct float64)
products(product_id int64, product_name object, category object, list_price_usd float64, unit_cost_usd float64)
customers(customer_id int64, signup_date object, country object, segment object, acquisition_channel object, is_test bool)
returns(return_id int64, order_id int64, product_id int64, return_date object, qty_returned int64, reason object)
fx_rates_monthly(month object, currency object, per_usd float64, usd_per_unit float64)


**2a.** Ask the AI:
> *Using this schema, write one DuckDB SQL query that shows, for each product category, total units sold, total revenue (quantity × unit_price) and total units returned, for completed USD orders.*

Paste and run it. Then run the **grain check** in the next cell before you believe it.

In [6]:
# The typical AI answer joins returns directly onto order_items:
ai_sql = """
SELECT p.category,
       SUM(oi.quantity)                 AS units_sold,
       SUM(oi.quantity * oi.unit_price) AS revenue,
       SUM(r.qty_returned)              AS units_returned
FROM order_items oi
JOIN orders o   ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
LEFT JOIN returns r ON r.order_id = oi.order_id AND r.product_id = oi.product_id
WHERE o.status = 'Completed' AND o.currency = 'USD'
GROUP BY p.category
ORDER BY revenue DESC
"""
q(ai_sql)

,category,units_sold,revenue,units_returned
0,Apparel,10170.0,1158986.30,862.0
1,Footwear,6255.0,963802.45,787.0
2,Tents,2700.0,910733.00,153.0
3,Backpacks,4762.0,600700.38,235.0
4,Accessories,4309.0,446675.91,169.0
5,Camp Kitchen,4360.0,197312.40,104.0


In [7]:
# Grain check: revenue computed WITHOUT touching returns. Does it match the AI's revenue column?
q("""
SELECT p.category, SUM(oi.quantity) AS units_sold, SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items oi
JOIN orders o   ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.status = 'Completed' AND o.currency = 'USD'
GROUP BY p.category
ORDER BY revenue DESC
""")

,category,units_sold,revenue
0,Apparel,9917.0,1133707.83
1,Footwear,6019.0,927260.81
2,Tents,2650.0,895033.50
3,Backpacks,4685.0,592878.15
4,Accessories,4281.0,443982.19
5,Camp Kitchen,4335.0,196347.65


If the numbers differ, you've found a **join fan-out**: `returns` has more than one row for some (order, product) pairs, so those sales get counted twice. Prove it:

In [8]:
q("""
SELECT COUNT(*) AS return_rows,
       COUNT(DISTINCT (order_id, product_id)) AS distinct_lines
FROM returns
""")

,return_rows,distinct_lines
0,3258,2733


**2b.** Fix the query: aggregate `returns` to one row per (order_id, product_id) in a **CTE** (`WITH r AS (...)`) *before* joining. Your revenue must match the grain check exactly.

In [9]:
q("""
WITH r AS (
    SELECT order_id, product_id, SUM(qty_returned) AS qty_returned
    FROM returns
    GROUP BY order_id, product_id
)
SELECT p.category,
       SUM(oi.quantity)                 AS units_sold,
       SUM(oi.quantity * oi.unit_price) AS revenue,
       SUM(r.qty_returned)              AS units_returned,
       ROUND(100.0 * SUM(r.qty_returned) / SUM(oi.quantity), 1) AS return_rate_pct
FROM order_items oi
JOIN orders o   ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
LEFT JOIN r     ON r.order_id = oi.order_id AND r.product_id = oi.product_id
WHERE o.status = 'Completed' AND o.currency = 'USD'
GROUP BY p.category
ORDER BY revenue DESC
""")

,category,units_sold,revenue,units_returned,return_rate_pct
0,Apparel,9917.0,1133707.83,862.0,8.7
1,Footwear,6019.0,927260.81,787.0,13.1
2,Tents,2650.0,895033.50,153.0,5.8
3,Backpacks,4685.0,592878.15,235.0,5.0
4,Accessories,4281.0,443982.19,169.0,3.9
5,Camp Kitchen,4335.0,196347.65,104.0,2.4


## Part 3: INNER vs LEFT JOIN (10 min)
Marketing asks: *"How many orders does the average customer place?"*

**3a.** Predict: will these two queries give the same answer? Then run them.

In [10]:
inner = q("""
SELECT AVG(n) AS avg_orders FROM (
    SELECT c.customer_id, COUNT(o.order_id) AS n
    FROM customers c JOIN orders o ON o.customer_id = c.customer_id
    WHERE NOT c.is_test
    GROUP BY c.customer_id)
""")
left = q("""
SELECT AVG(n) AS avg_orders FROM (
    SELECT c.customer_id, COUNT(o.order_id) AS n
    FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE NOT c.is_test
    GROUP BY c.customer_id)
""")
print("INNER:", inner.avg_orders[0].round(2), "  LEFT:", left.avg_orders[0].round(2))

INNER: 7.87   LEFT: 6.47


**3b.** In one sentence each: which customers does each query include? Which one answers Marketing's question? (It depends on the question they *meant*, so write the clarifying question you'd send back.)

## Part 4: Window functions, the "top N per group" pattern (10 min)
**4a.** Ask the AI for: *the single best-selling product (by units, completed orders) in each country*. It should use `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`. Run it and check that you get exactly one row per country.

In [11]:
q("""
WITH units AS (
    SELECT o.country, p.product_name, SUM(oi.quantity) AS units
    FROM order_items oi
    JOIN orders o   ON o.order_id = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.status = 'Completed' AND NOT c.is_test
    GROUP BY o.country, p.product_name
)
SELECT country, product_name, units
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY country ORDER BY units DESC) AS rk FROM units)
WHERE rk = 1
ORDER BY units DESC
""")

,country,product_name,units
0,United States,Down Puffy Jacket,1302.0
1,United Kingdom,Sun Hoodie,401.0
2,Germany,Merino Base Layer Top,342.0
3,France,Merino Base Layer Top,169.0
4,Netherlands,Trail Pant,102.0
5,Ireland,Trail Pant,102.0


### Exit ticket (submit)
Write the **three checks** you will run on any AI-written SQL query before you send its numbers to anyone. (Hint: row counts, a total you can reproduce another way, and ___.)